# WLASL Landmark Extraction — Multi-Session Kaggle Pipeline

Run this notebook on Kaggle to extract hand/pose/face landmarks from WLASL videos.  
Because Kaggle sessions are limited to ~12h, this notebook supports **multi-session resumption**:

1. **Session 1**: Start fresh, extract as many samples as possible, then run Cell 4 to export progress.
2. **Session 2+**: Attach the previous session's output dataset, run Cell 1 to restore progress, then Cell 2 continues from where it left off.

---
## Setup
- **Input datasets required**:
  - `wlasl-complete` (WLASL videos + `WLASL_v0.3.json`)
  - *(optional for resume)* previous session output dataset, e.g. `last-hope-s1`
- **Output**: `landmarks_npy/`, `meta/`, then zipped as `landmarks_master.zip` for the next session.


## Cell 0 — Global configuration

In [ ]:
import os, glob

# ─── Paths ────────────────────────────────────────────────────────────────────
WORKING_DIR        = "/kaggle/working"
INPUT_DIR          = "/kaggle/input"

# Where we store extracted .npy files during this session
ENCODE_DIR         = os.path.join(WORKING_DIR, "landmarks_npy")
# Where we store meta files (manifest, parsed data, labels)
META_DIR           = os.path.join(WORKING_DIR, "meta")

# Meta file paths
MANIFEST_PATH              = os.path.join(META_DIR, "processing_manifest.json")
PARSED_DATA_PATH           = os.path.join(META_DIR, "WLASL_parsed_data.json")
LABELS_NPZ_PATH            = os.path.join(META_DIR, "labels.npz")
FILTERED_LABELS_TXT_PATH   = os.path.join(META_DIR, "filtered_labels.txt")

# ─── WLASL source data ────────────────────────────────────────────────────────
# Adjust dataset slug as needed (the folder name under /kaggle/input/)
WLASL_DATASET      = "wlasl-complete"
WLASL_JSON         = os.path.join(INPUT_DIR, WLASL_DATASET, "WLASL_v0.3.json")
VIDEO_DIR_PRIMARY  = os.path.join(INPUT_DIR, WLASL_DATASET, "videos")
VIDEO_DIR_BACKUP   = os.path.join(INPUT_DIR, WLASL_DATASET, "missing")

# ─── Previous-session dataset ─────────────────────────────────────────────────
# Kaggle can mount datasets under two different paths depending on the source:
#   Standard (your own dataset):  /kaggle/input/{slug}/
#   Non-standard (another user's dataset):  /kaggle/input/datasets/{owner}/{slug}/
#
# Option A — auto-detect (recommended):
#   Set PREV_SESSION_SLUG to the dataset slug and leave PREV_DATASET_DIR as "".
#   The notebook will search both standard and non-standard mount points.
#
# Option B — override path directly (use when auto-detect fails):
#   Set PREV_DATASET_DIR to the full path, e.g.:
#     PREV_DATASET_DIR = "/kaggle/input/datasets/pandk24/last-hope-s1"
#
PREV_SESSION_SLUG = "last-hope-s1"   # <── edit per session (or set "" for session 1)
PREV_DATASET_DIR  = ""               # <── override with full path if auto-detect fails
                                     # e.g. "/kaggle/input/datasets/pandk24/last-hope-s1"


def _find_dataset_dir(slug, input_root="/kaggle/input"):
    """Resolve the mounted directory for a dataset slug.

    Kaggle mounts datasets at one of:
      1. /kaggle/input/{slug}/                        (standard – your own datasets)
      2. /kaggle/input/datasets/{owner}/{slug}/       (non-standard – other users' datasets)

    Returns the first matching directory, or None if not found.
    """
    if not slug or not os.path.isdir(input_root):
        return None

    # 1. Standard path
    direct = os.path.join(input_root, slug)
    if os.path.isdir(direct):
        return direct

    # 2. Under datasets/{owner}/  (any owner)
    for path in glob.glob(os.path.join(input_root, "datasets", "*", slug)):
        if os.path.isdir(path):
            return path

    return None


# Resolve PREV_DATASET_DIR
if not PREV_DATASET_DIR:
    if PREV_SESSION_SLUG:
        PREV_DATASET_DIR = _find_dataset_dir(PREV_SESSION_SLUG, INPUT_DIR) or ""
        if PREV_DATASET_DIR:
            print(f"[CONFIG] Auto-detected PREV_DATASET_DIR: {PREV_DATASET_DIR}")
        else:
            _top = os.listdir(INPUT_DIR) if os.path.isdir(INPUT_DIR) else []
            _datasets_dir = os.path.join(INPUT_DIR, "datasets")
            _owners = os.listdir(_datasets_dir) if os.path.isdir(_datasets_dir) else []
            print(f"[CONFIG] WARNING: dataset slug '{PREV_SESSION_SLUG}' not found under {INPUT_DIR}")
            print(f"[CONFIG] Top-level inputs       : {_top}")
            print(f"[CONFIG] Owners under datasets/ : {_owners}")
            print(f"[CONFIG] → If the dataset is attached, set PREV_DATASET_DIR to its full path.")
    else:
        print("[CONFIG] No previous session configured (PREV_SESSION_SLUG is empty).")
else:
    print(f"[CONFIG] Using PREV_DATASET_DIR: {PREV_DATASET_DIR}")

# ─── Session time budget ──────────────────────────────────────────────────────
SECONDS_PER_HOUR       = 3600
SESSION_SECONDS_LIMIT  = 11.5 * SECONDS_PER_HOUR  # usable session wall-time
RESERVE_ZIP_SECONDS    = 30 * 60                   # reserve time at end for zipping

# Ensure key directories exist
os.makedirs(ENCODE_DIR, exist_ok=True)
os.makedirs(META_DIR,   exist_ok=True)

print("[CONFIG] ENCODE_DIR      :", ENCODE_DIR)
print("[CONFIG] META_DIR        :", META_DIR)
print("[CONFIG] PREV_DATASET_DIR:", PREV_DATASET_DIR or "(none)")


## Cell 1 — Restore progress from previous session

This cell handles **two source formats**:
1. `landmarks_master.zip` inside the previous dataset (packed by Cell 4)
2. An already-extracted folder structure (`landmarks_npy/` + meta files) inside the previous dataset

After running you should see non-zero `npy in working` if a previous session exists.

**Non-standard paths**: if your previous dataset is mounted at `/kaggle/input/datasets/{owner}/{slug}/` rather than the default `/kaggle/input/{slug}/`, set `PREV_DATASET_DIR` in Cell 0 to the full path, or let auto-detection handle it.

In [ ]:
import os, glob, shutil, zipfile, json

# ─────────────────────────────────────────────────────────────────────────────
# Helper: recursively locate all .npy files under a root directory
# ─────────────────────────────────────────────────────────────────────────────
def _find_npy_files(root):
    return glob.glob(os.path.join(root, "**", "*.npy"), recursive=True)

# ─────────────────────────────────────────────────────────────────────────────
# Helper: copy meta files (manifest, parsed data, labels) from src_dir to META_DIR
# ─────────────────────────────────────────────────────────────────────────────
META_FILENAMES = [
    "processing_manifest.json",
    "WLASL_parsed_data.json",
    "labels.npz",
    "filtered_labels.txt",
]

_RESTORE_TMP_DIR = os.path.join("/kaggle/working", "_restore_tmp")

def _copy_meta_files(src_dir, meta_dir):
    """Recursively search src_dir for meta files and copy to meta_dir."""
    copied = []
    for fname in META_FILENAMES:
        # direct path first
        direct = os.path.join(src_dir, fname)
        if os.path.exists(direct):
            shutil.copy2(direct, os.path.join(meta_dir, fname))
            copied.append(fname)
            continue
        # recursive search
        found = glob.glob(os.path.join(src_dir, "**", fname), recursive=True)
        if found:
            shutil.copy2(found[0], os.path.join(meta_dir, fname))
            copied.append(fname)
    return copied

# ─────────────────────────────────────────────────────────────────────────────
# Main restore logic
# ─────────────────────────────────────────────────────────────────────────────
def restore_progress(prev_dataset_dir, encode_dir, meta_dir):
    """
    Restore .npy files and meta files from a previous session dataset into
    the current working directories.

    Accepts two source formats:
      1. landmarks_master.zip  (produced by Cell 4)
      2. Folder structure with landmarks_npy/ + meta files at top level

    The prev_dataset_dir may be at either:
      - /kaggle/input/{slug}/                     (standard)
      - /kaggle/input/datasets/{owner}/{slug}/    (non-standard)

    Returns (npy_copied, meta_copied) counts.
    """
    if not prev_dataset_dir:
        print("[RESTORE] PREV_DATASET_DIR is empty — skipping restore.")
        return 0, []

    if not os.path.isdir(prev_dataset_dir):
        # Provide a helpful diagnostic with all available paths
        input_root = "/kaggle/input"
        top_level  = os.listdir(input_root) if os.path.isdir(input_root) else []
        datasets_dir = os.path.join(input_root, "datasets")
        nested = glob.glob(os.path.join(datasets_dir, "*", "*")) if os.path.isdir(datasets_dir) else []
        raise FileNotFoundError(
            f"[RESTORE] Previous dataset directory not found: {prev_dataset_dir}\n"
            f"  Top-level inputs ({input_root}): {top_level}\n"
            f"  Non-standard mounts (datasets/*/): {nested}\n"
            f"  → Check PREV_SESSION_SLUG / PREV_DATASET_DIR in Cell 0 and "
            f"make sure the dataset is attached in the Kaggle input panel."
        )

    print(f"[RESTORE] Searching in: {prev_dataset_dir}")
    print(f"[RESTORE] Contents: {os.listdir(prev_dataset_dir)}")

    # ── Determine extraction root ────────────────────────────────────────────
    zip_path    = os.path.join(prev_dataset_dir, "landmarks_master.zip")
    extract_root = None

    if os.path.exists(zip_path):
        print(f"[RESTORE] Found landmarks_master.zip — extracting …")
        extract_root = _RESTORE_TMP_DIR
        os.makedirs(extract_root, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_root)
        print(f"[RESTORE] Extracted to: {extract_root}")
    else:
        # Try using the prev_dataset_dir directly as the extraction root
        extract_root = prev_dataset_dir
        print(f"[RESTORE] No landmarks_master.zip — treating folder as extraction root.")

    # ── Locate and copy .npy files ───────────────────────────────────────────
    npy_files = _find_npy_files(extract_root)
    print(f"[RESTORE] Discovered {len(npy_files)} .npy file(s) under {extract_root}")
    if not npy_files:
        print(f"[RESTORE] WARNING: No .npy files found. "
              f"Verify that '{prev_dataset_dir}' is the correct previous-session dataset.")

    npy_copied = 0
    for src in npy_files:
        dst = os.path.join(encode_dir, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            npy_copied += 1
    print(f"[RESTORE] Copied {npy_copied} new .npy file(s) → {encode_dir}")

    # ── Locate and copy meta files ───────────────────────────────────────────
    meta_copied = _copy_meta_files(extract_root, meta_dir)
    print(f"[RESTORE] Meta files copied: {meta_copied}")

    # ── Final diagnostics ────────────────────────────────────────────────────
    npy_in_working   = len(glob.glob(os.path.join(encode_dir, "*.npy")))
    manifest_exists  = os.path.exists(os.path.join(meta_dir, "processing_manifest.json"))
    print(f"[RESTORE] npy in working : {npy_in_working}")
    print(f"[RESTORE] manifest exists: {manifest_exists}")
    if manifest_exists:
        with open(os.path.join(meta_dir, "processing_manifest.json")) as f:
            m = json.load(f)
        print(f"[RESTORE] manifest processed_ids: {len(set(m.get('processed_ids', [])))}")

    # Clean up temp extraction directory if we created one
    tmp_dir = _RESTORE_TMP_DIR
    if os.path.isdir(tmp_dir) and extract_root == tmp_dir:
        shutil.rmtree(tmp_dir)

    return npy_copied, meta_copied


# ── Run restore ──────────────────────────────────────────────────────────────
npy_restored, meta_restored = restore_progress(PREV_DATASET_DIR, ENCODE_DIR, META_DIR)


## Cell 1b — Verify restore (sanity check)

If `npy in working` shows 0 and you expected a previous session, re-check:
- `PREV_SESSION_SLUG` in Cell 0 matches the dataset slug exactly
- The dataset is attached in the Kaggle input panel
- If the dataset was added from another user, it may be mounted at `/kaggle/input/datasets/{owner}/{slug}` — set `PREV_DATASET_DIR` in Cell 0 to that path
- The previous session ran Cell 4 to produce `landmarks_master.zip` **or** the folder contains `landmarks_npy/`

In [ ]:
import os, glob, json

npy_working   = glob.glob(os.path.join(ENCODE_DIR, "*.npy"))
manifest_path = MANIFEST_PATH

print("=== Post-restore sanity check ===")
print(f"npy in working   : {len(npy_working)}")
print(f"manifest exists  : {os.path.exists(manifest_path)}")
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        m = json.load(f)
    print(f"manifest processed_ids : {len(set(m.get('processed_ids', [])))}")
    print(f"manifest total_items   : {m.get('total_items')}")
print(f"parsed data exists : {os.path.exists(PARSED_DATA_PATH)}")
print(f"labels.npz exists  : {os.path.exists(LABELS_NPZ_PATH)}")


## Cell 2 — Extract landmarks (anti-reset resume)

- Builds `done` as the **union** of disk .npy files and manifest `processed_ids`.
- Prints `[RESUME]` and `[GLOBAL START]` before processing so you can confirm the correct starting point.
- Processes only `pending` samples (those not already done).
- Saves manifest every 25 samples.


In [ ]:
import os, json, time, glob, gc
import numpy as np
from tqdm import tqdm
import cv2
import mediapipe as mp
from concurrent.futures import ThreadPoolExecutor

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def load_manifest(manifest_path, total_items):
    if os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            m = json.load(f)
    else:
        m = {"total_items": total_items, "processed_ids": [], "failed_ids": [], "last_update_time": None}
    m["total_items"]    = total_items
    m["processed_ids"]  = list(set(m.get("processed_ids", [])))
    m["failed_ids"]     = list(set(m.get("failed_ids", [])))
    return m

def save_manifest(manifest, manifest_path):
    manifest["last_update_time"] = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

def print_global_progress(total_items, done_count, prefix="GLOBAL"):
    pct = (done_count / total_items * 100) if total_items else 0.0
    print(f"[{prefix}] {done_count}/{total_items} ({pct:.2f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# Parse / load WLASL data
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(PARSED_DATA_PATH):
    with open(PARSED_DATA_PATH, "r") as f:
        data = json.load(f)
    print(f"[DATA] Loaded cached parsed data: {len(data)} items")
else:
    with open(WLASL_JSON, "r") as f:
        all_data = json.load(f)

    data = []
    for entry in tqdm(all_data, ncols=100, desc="Parsing WLASL JSON"):
        gloss = entry["gloss"]
        for ins in entry["instances"]:
            vid = ins["video_id"]
            p1  = os.path.join(VIDEO_DIR_PRIMARY, f"{vid}.mp4")
            p2  = os.path.join(VIDEO_DIR_BACKUP,  f"{vid}.mp4")
            if os.path.exists(p1):
                vpath = p1
            elif os.path.exists(p2):
                vpath = p2
            else:
                continue
            data.append({
                "gloss":       gloss,
                "video_path":  vpath,
                "frame_start": ins["frame_start"],
                "frame_end":   ins["frame_end"],
                "split":       ins["split"],
            })
    os.makedirs(META_DIR, exist_ok=True)
    with open(PARSED_DATA_PATH, "w") as f:
        json.dump(data, f, indent=2)
    print(f"[DATA] Parsed and saved: {len(data)} items")

total_items = len(data)

# ─────────────────────────────────────────────────────────────────────────────
# Landmark configuration
# ─────────────────────────────────────────────────────────────────────────────

filtered_hand = list(range(21))
filtered_pose = [11, 12, 13, 14, 15, 16]
filtered_face = [
    0, 4, 7, 8, 10, 13, 14, 17, 21, 33, 37, 39, 40, 46, 52, 53, 54, 55, 58,
    61, 63, 65, 66, 67, 70, 78, 80, 81, 82, 84, 87, 88, 91, 93, 95, 103, 105,
    107, 109, 127, 132, 133, 136, 144, 145, 146, 148, 149, 150, 152, 153, 154,
    155, 157, 158, 159, 160, 161, 162, 163, 172, 173, 176, 178, 181, 185, 191,
    234, 246, 249, 251, 263, 267, 269, 270, 276, 282, 283, 284, 285, 288, 291,
    293, 295, 296, 297, 300, 308, 310, 311, 312, 314, 317, 318, 321, 323, 324,
    332, 334, 336, 338, 356, 361, 362, 365, 373, 374, 375, 377, 378, 379, 380,
    381, 382, 384, 385, 386, 387, 388, 389, 390, 397, 398, 400, 402, 405, 409,
    415, 454, 466, 468, 473,
]

HAND_NUM  = len(filtered_hand)   # 21
POSE_NUM  = len(filtered_pose)   # 6
RIGHT_HAND_INDEX     = 0         # mediapipe hand index for right hand
MANIFEST_SAVE_INTERVAL = 25      # save manifest every N processed samples
FACE_NUM  = len(filtered_face)
NUM_LANDMARKS = HAND_NUM * 2 + POSE_NUM + FACE_NUM

hands     = mp.solutions.hands.Hands()
pose      = mp.solutions.pose.Pose()
face_mesh = mp.solutions.face_mesh.FaceMesh(refine_landmarks=True)

def get_frame_landmarks(frame_rgb):
    all_lm = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)

    def _hands():
        r = hands.process(frame_rgb)
        if r.multi_hand_landmarks:
            for i, h in enumerate(r.multi_hand_landmarks):
                arr = np.array([(lm.x, lm.y, lm.z) for lm in h.landmark], dtype=np.float32)
                is_right = (r.multi_handedness[i].classification[0].index == RIGHT_HAND_INDEX)
                if is_right:
                    all_lm[:HAND_NUM] = arr
                else:
                    all_lm[HAND_NUM:HAND_NUM * 2] = arr

    def _pose():
        r = pose.process(frame_rgb)
        if r.pose_landmarks:
            arr = np.array([(lm.x, lm.y, lm.z) for lm in r.pose_landmarks.landmark], dtype=np.float32)
            all_lm[HAND_NUM * 2:HAND_NUM * 2 + POSE_NUM] = arr[filtered_pose]

    def _face():
        r = face_mesh.process(frame_rgb)
        if r.multi_face_landmarks:
            arr = np.array([(lm.x, lm.y, lm.z) for lm in r.multi_face_landmarks[0].landmark], dtype=np.float32)
            all_lm[HAND_NUM * 2 + POSE_NUM:] = arr[filtered_face]

    with ThreadPoolExecutor(max_workers=3) as ex:
        ex.submit(_hands)
        ex.submit(_pose)
        ex.submit(_face)
    return all_lm


def get_video_landmarks(video_path, start_frame=1, end_frame=-1):
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if start_frame <= 1:
        start_frame = 1
    elif start_frame > total:
        start_frame, end_frame = 1, total
    if end_frame < 0:
        end_frame = total

    n   = max(0, end_frame - start_frame + 1)
    out = np.zeros((n, NUM_LANDMARKS, 3), dtype=np.float32)

    idx = 1
    while cap.isOpened() and idx <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        if idx >= start_frame:
            frame.flags.writeable = False
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            out[idx - start_frame] = get_frame_landmarks(rgb)
        idx += 1

    cap.release()
    hands.reset()
    pose.reset()
    face_mesh.reset()
    return out

# ─────────────────────────────────────────────────────────────────────────────
# Anti-reset resume: build done set from DISK first, then union with manifest
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(ENCODE_DIR, exist_ok=True)
manifest = load_manifest(MANIFEST_PATH, total_items)

disk_done     = set(os.path.basename(x).replace(".npy", "")
                    for x in glob.glob(os.path.join(ENCODE_DIR, "*.npy")))
manifest_done = set(manifest.get("processed_ids", []))
done          = disk_done | manifest_done

print(f"[RESUME] disk_done={len(disk_done)}, manifest_done={len(manifest_done)}, union={len(done)}")
print_global_progress(total_items, len(done), prefix="GLOBAL START")

# Sync manifest if disk has more progress than manifest recorded
if len(disk_done) > len(manifest_done):
    manifest["processed_ids"] = list(done)
    save_manifest(manifest, MANIFEST_PATH)
    print("[RESUME] Manifest synced from disk_done.")

# ─────────────────────────────────────────────────────────────────────────────
# Session time budget and pending list
# ─────────────────────────────────────────────────────────────────────────────

start_time = time.time()
deadline   = start_time + SESSION_SECONDS_LIMIT - RESERVE_ZIP_SECONDS
print(f"[SESSION] processing budget ~ {(SESSION_SECONDS_LIMIT - RESERVE_ZIP_SECONDS) / 3600:.2f}h")

pending = [str(i) for i in range(total_items) if str(i) not in done]
print(f"[PENDING] {len(pending)} sample(s) remaining")

# ─────────────────────────────────────────────────────────────────────────────
# Main encoding loop
# ─────────────────────────────────────────────────────────────────────────────

pbar = tqdm(pending, ncols=100, desc="Encoding landmarks")
for idx_str in pbar:
    if time.time() >= deadline:
        print("\n[STOP] Reached time budget — move to Cell 4 to pack/export.")
        break

    i        = int(idx_str)
    npy_path = os.path.join(ENCODE_DIR, f"{idx_str}.npy")

    if os.path.exists(npy_path):
        done.add(idx_str)
        continue

    try:
        item = data[i]
        arr  = get_video_landmarks(item["video_path"], item["frame_start"], item["frame_end"])
        np.save(npy_path, arr)
        done.add(idx_str)

        if len(done) % MANIFEST_SAVE_INTERVAL == 0:
            manifest["processed_ids"] = list(done)
            save_manifest(manifest, MANIFEST_PATH)

        gp = len(done) * 100.0 / total_items
        pbar.set_postfix({"global_%": f"{gp:.2f}"})

    except Exception as e:
        failed = set(manifest.get("failed_ids", []))
        failed.add(idx_str)
        manifest["failed_ids"] = list(failed)
        save_manifest(manifest, MANIFEST_PATH)
        continue

# Final manifest save
manifest["processed_ids"] = list(done)
save_manifest(manifest, MANIFEST_PATH)

print_global_progress(total_items, len(done), prefix="FINAL PROGRESS")
print(f"[INFO] failed_ids={len(manifest.get('failed_ids', []))}")

gc.collect()


## Cell 3 — Label encoding (cached)

Builds `labels.npz` and `filtered_labels.txt` from processed samples.  
**Skipped automatically** if outputs already exist.

In [ ]:
import os, json, glob
import numpy as np

# ── Skip rebuild if already exists ──────────────────────────────────────────
labels_ok   = os.path.exists(LABELS_NPZ_PATH)   and os.path.getsize(LABELS_NPZ_PATH)   > 1024
filtered_ok = os.path.exists(FILTERED_LABELS_TXT_PATH) and os.path.getsize(FILTERED_LABELS_TXT_PATH) > 10

if labels_ok and filtered_ok:
    print("[LABEL] labels.npz and filtered_labels.txt already exist — skipping rebuild.")
else:
    try:
        import fasttext
        import fasttext.util
        fasttext.util.download_model("en", if_exists="ignore")
        ft = fasttext.load_model("cc.en.300.bin")
    except ImportError:
        raise ImportError("fasttext not installed. Run: pip install fasttext-wheel")

    with open(PARSED_DATA_PATH, "r") as f:
        parsed = json.load(f)

    # Only include labels for samples that have been processed
    processed_npy = set(
        os.path.basename(x).replace(".npy", "")
        for x in glob.glob(os.path.join(ENCODE_DIR, "*.npy"))
    )

    uniq_labels = sorted(set(
        item["gloss"]
        for i, item in enumerate(parsed)
        if item["split"] == "train" and str(i) in processed_npy
    ))
    print(f"[LABEL] Building labels.npz for {len(uniq_labels)} labels …")

    encoded_labels = {lab: ft.get_word_vector(lab) for lab in uniq_labels}
    os.makedirs(META_DIR, exist_ok=True)
    np.savez_compressed(LABELS_NPZ_PATH, **encoded_labels)
    print(f"[LABEL] Saved: {LABELS_NPZ_PATH}")

    with open(FILTERED_LABELS_TXT_PATH, "w") as f:
        f.write("\n".join(uniq_labels))
    print(f"[LABEL] Saved: {FILTERED_LABELS_TXT_PATH}")


## Cell 4 — Pack & export for next session

Creates:
- `landmarks_master.zip` — all `.npy` files (used by Cell 1 restore in the next session)
- `session_output.zip` — everything in `meta/` + `landmarks_master.zip`

**Run this cell before the session times out.**  
Then upload `session_output.zip` as a new Kaggle dataset to use in the next session.

In [ ]:
import os, glob, zipfile, time

LANDMARKS_ZIP = os.path.join(WORKING_DIR, "landmarks_master.zip")
SESSION_ZIP   = os.path.join(WORKING_DIR, "session_output.zip")

# ── Pack landmarks_master.zip ────────────────────────────────────────────────
npy_files = sorted(glob.glob(os.path.join(ENCODE_DIR, "*.npy")))
print(f"[PACK] Packing {len(npy_files)} .npy files into landmarks_master.zip …")
t0 = time.time()
with zipfile.ZipFile(LANDMARKS_ZIP, "w", compression=zipfile.ZIP_STORED) as zf:
    for npy in npy_files:
        zf.write(npy, arcname=os.path.join("landmarks_npy", os.path.basename(npy)))
    # Also include meta files inside the zip for self-contained restore
    for fname in ["processing_manifest.json", "WLASL_parsed_data.json", "labels.npz", "filtered_labels.txt"]:
        fpath = os.path.join(META_DIR, fname)
        if os.path.exists(fpath):
            zf.write(fpath, arcname=fname)
print(f"[PACK] landmarks_master.zip ready ({os.path.getsize(LANDMARKS_ZIP)/1024/1024:.1f} MB, {time.time()-t0:.1f}s)")

# ── Pack session_output.zip ──────────────────────────────────────────────────
print("[PACK] Creating session_output.zip …")
with zipfile.ZipFile(SESSION_ZIP, "w", compression=zipfile.ZIP_STORED) as zf:
    # landmarks_master.zip
    zf.write(LANDMARKS_ZIP, arcname="landmarks_master.zip")
    # individual meta files at top level (for direct-folder restore)
    for fname in ["processing_manifest.json", "WLASL_parsed_data.json", "labels.npz", "filtered_labels.txt"]:
        fpath = os.path.join(META_DIR, fname)
        if os.path.exists(fpath):
            zf.write(fpath, arcname=fname)
print(f"[PACK] session_output.zip ready ({os.path.getsize(SESSION_ZIP)/1024/1024:.1f} MB)")
print()
print("Next steps:")
print("  1. Download session_output.zip from /kaggle/working")
print("  2. Create a new Kaggle dataset from session_output.zip")
print("  3. In the next session, set PREV_SESSION_SLUG to that dataset slug")
print("     If Kaggle mounts it under /kaggle/input/datasets/{owner}/{slug}/,")
print("     set PREV_DATASET_DIR to the full path in Cell 0.")
